# Module 5.1: Hierarchical Hidden Markov Models

## Learning Objectives
By completing this notebook, you will:
1. Understand hierarchical HMM structure and motivation
2. Implement two-level HHMM with super-states and sub-states
3. Apply to multi-timescale financial patterns
4. Compare HHMM vs. flat HMM performance
5. Interpret hierarchical regime structure

## Prerequisites
- Standard HMM algorithms (Module 2)
- Gaussian HMMs (Module 3)

## Estimated Time: 60 minutes

---

In [ ]:
import sys; sys.path.insert(0, '../../../../..')
from resources.notebook_style import apply_course_theme, learning_objectives, section_divider, callout, key_takeaways
from resources.graphics.plot_theme import apply_plot_theme

In [ ]:
apply_course_theme()
apply_plot_theme()

In [ ]:
learning_objectives([
    "Standard HMM algorithms (Module 2)",
    "Gaussian HMMs (Module 3)"
])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 1. HHMM Motivation

Standard HMMs: flat state space  
**Problem**: Real systems have multi-scale structure

**Example**: Markets
- **Super-states** (long-term): Economic cycle (expansion, recession)
- **Sub-states** (short-term): Daily volatility (low, high)

In [ ]:
def generate_hierarchical_data(T=1000):
    # Level 1: Super-states (slow-moving economic regimes)
    super_A = np.array([[0.99, 0.01], [0.02, 0.98]])
    super_pi = np.array([0.7, 0.3])
    super_states = np.zeros(T, dtype=int)
    super_states[0] = np.random.choice(2, p=super_pi)
    for t in range(1,T):
        super_states[t] = np.random.choice(2, p=super_A[super_states[t-1]])
    
    # Level 2: Sub-states (fast-moving volatility)
    # Each super-state has its own sub-state dynamics
    sub_A = [
        np.array([[0.9, 0.1], [0.3, 0.7]]),  # Expansion: mostly low-vol
        np.array([[0.6, 0.4], [0.4, 0.6]])   # Recession: mixed vol
    ]
    sub_states = np.zeros(T, dtype=int)
    sub_states[0] = 0
    
    for t in range(1, T):
        super_s = super_states[t]
        sub_states[t] = np.random.choice(2, p=sub_A[super_s][sub_states[t-1]])
    
    # Emissions depend on both levels
    emission_params = {
        (0, 0): {'mu': 0.0006, 'sig': 0.008},   # Expansion + Low-Vol
        (0, 1): {'mu': 0.0008, 'sig': 0.015},   # Expansion + High-Vol
        (1, 0): {'mu': -0.0005, 'sig': 0.012},  # Recession + Low-Vol
        (1, 1): {'mu': -0.0010, 'sig': 0.025}   # Recession + High-Vol
    }
    
    obs = np.array([np.random.normal(emission_params[(super_states[t], sub_states[t])]['mu'],
                                     emission_params[(super_states[t], sub_states[t])]['sig']) 
                    for t in range(T)])
    
    return obs, super_states, sub_states

obs, super_s, sub_s = generate_hierarchical_data(1000)
print(f'Generated {len(obs)} observations')
print(f'Super-state distribution: {np.bincount(super_s)}')
print(f'Sub-state distribution: {np.bincount(sub_s)}')

## 2. HHMM Implementation

In [ ]:
class HierarchicalHMM:
    def __init__(self, n_super_states, n_sub_states):
        self.K_super = n_super_states
        self.K_sub = n_sub_states
        self.K_flat = n_super_states * n_sub_states
        
        # Super-state transition
        self.A_super = np.random.rand(self.K_super, self.K_super)
        self.A_super /= self.A_super.sum(axis=1, keepdims=True)
        
        # Sub-state transitions (one per super-state)
        self.A_sub = [np.random.rand(self.K_sub, self.K_sub) for _ in range(self.K_super)]
        for i in range(self.K_super):
            self.A_sub[i] /= self.A_sub[i].sum(axis=1, keepdims=True)
        
        # Emissions (for each combination)
        self.means = np.random.randn(self.K_flat) * 0.001
        self.stds = np.ones(self.K_flat) * 0.01
    
    def _flat_state(self, super_s, sub_s):
        return super_s * self.K_sub + sub_s
    
    def _to_hierarchical(self, flat_s):
        return flat_s // self.K_sub, flat_s % self.K_sub
    
    def fit(self, obs, max_iter=30):
        # Simplified EM for HHMM
        # (Full implementation would use hierarchical forward-backward)
        print(f'Fitting HHMM with {self.K_super} super-states, {self.K_sub} sub-states')
        print('(Simplified implementation)')  
        return self
    
    def predict(self, obs):
        # Simplified Viterbi
        # Returns (super_states, sub_states)
        T = len(obs)
        super_pred = np.zeros(T, dtype=int)
        sub_pred = np.zeros(T, dtype=int)
        # Placeholder: use flat HMM approximation
        return super_pred, sub_pred

hhmm = HierarchicalHMM(n_super_states=2, n_sub_states=2)
print(hhmm.fit(obs))

## Exercise 1: Implement Hierarchical Viterbi

**Task**: Complete the predict method for HHMM

**Expected Output**: Super-state and sub-state sequences

In [ ]:
# YOUR CODE HERE
def hierarchical_viterbi(hhmm, observations):
    # Your implementation
    return None, None

super_decoded, sub_decoded = hierarchical_viterbi(hhmm, obs)

In [ ]:
def test_exercise_1():
    assert super_decoded is not None
    assert sub_decoded is not None
    print('✅ Exercise 1 passed!')
test_exercise_1()

## Exercise 2: Compare Flat vs. Hierarchical

In [ ]:
# YOUR CODE HERE
def compare_flat_vs_hierarchical(obs, true_super, true_sub):
    # Fit both models and compare BIC
    return None

comparison = compare_flat_vs_hierarchical(obs, super_s, sub_s)

In [ ]:
def test_exercise_2():
    assert comparison is not None
    print('✅ Exercise 2 passed!')
test_exercise_2()

## Exercise 3: Multi-Timescale Analysis

In [ ]:
# YOUR CODE HERE
def analyze_timescales(super_states, sub_states):
    # Compute persistence at each level
    return None

timescale_analysis = analyze_timescales(super_s, sub_s)

In [ ]:
def test_exercise_3():
    assert timescale_analysis is not None
    print('✅ Exercise 3 passed!')
test_exercise_3()

## Summary

### Key Takeaways
1. HHMMs capture multi-timescale dynamics
2. Super-states = slow-moving regimes
3. Sub-states = fast-moving fluctuations
4. More parameters but better interpretability
5. Useful for macroeconomic + market microstructure

---